In [2]:
import sklearn
from sklearn.pipeline import Pipeline
from sklearn import preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import cross_validate, StratifiedKFold, KFold
import numpy as np
from sklearn.datasets import load_iris
import pandas as pd
import os
import re 
import datetime as dt
import sys 
import importlib
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import (
	OneHotEncoder,
	OrdinalEncoder,
	RobustScaler,
)
from sklearn.preprocessing import (
	PowerTransformer, 
	QuantileTransformer
)
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import PowerTransformer, StandardScaler, QuantileTransformer, Normalizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupShuffleSplit
from sklearn.model_selection import cross_validate
sklearn.set_config(enable_metadata_routing=False)
from sklearn.model_selection import train_test_split
from sklearn.compose import make_column_selector as selector
#from xgboost import XGBClassifier, XGBRegressor
import time 
import pandas as pd
import os
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import PowerTransformer, StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import make_scorer
from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.base import BaseEstimator, RegressorMixin
from scipy.stats import pearsonr
import plotly.express as px
from sklearn.linear_model import Ridge
from sklearn.decomposition import PCA
from sklearn.ensemble import HistGradientBoostingRegressor
import numpy as np
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score, mean_absolute_error

import shap
shap.initjs()
############ LOAD in custom packages ################

project_root = os.path.join(os.getcwd(), "..") # Get path of the project 
sys.path.append(project_root) # Add project root to sys.path for script usage

# Import and reload (optional) custom scripts
from scripts import paths
from scripts import preprocessing as pre
from scripts import visualization as vis
from scripts import variables
from scripts import feature_selection as fs

importlib.reload(paths)
importlib.reload(pre)
importlib.reload(vis)
importlib.reload(variables)
importlib.reload(fs)

# Filepaths
brighten_dir = paths.DATA


################ DEFINE column variables from data ###################
from scripts.variables import id_columns
from scripts.variables import all_cols, all_daily_cols, weekly_cols, baseline_cols, drop_weekly_cols
from scripts.variables import daily_cols_v1, daily_v2_sensor_hr, daily_v2_weather, daily_cols_v2 
from scripts.variables import gad_cols, phq9_base, alc_cols, phq9_cols, phq2_cols, sleep_cols, gic_cols, sds_cols


# Define label variables
df_names = ['v1_day', 'v2_day', 'v1_week', 'v2_week']
aggregate_dfs = ['alldays_df','week_df']
# Update endings list if order changes


import warnings
warnings.filterwarnings(
	"ignore",
	message="Skipping features without any observed values",
	category=UserWarning,
	module="sklearn.impute._base"
)


# Defining the transformations
yj_pipeline = Pipeline(steps=[
	('power', PowerTransformer(method='yeo-johnson')),
	('scale', StandardScaler())
])

bc_pipeline = Pipeline(steps=[
	('power', PowerTransformer(method='box-cox')),
	('scale', StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
	('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

ordinal_pipeline = Pipeline(steps=[
	('scale', StandardScaler())
])

non_skewed_pipeline = Pipeline(steps=[
	('scale', StandardScaler())
])


########################################## MODELS #######################################


# Models
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
# Dummy baseline model
class GroupMeanRegressor(BaseEstimator, RegressorMixin):
	def fit(self, X, y, groups):
		self.group_means_ = y.groupby(groups.squeeze()).mean().squeeze()
		self.global_mean_ = y.mean().squeeze()
		return self
	
	def predict(self, X, groups):
		return groups.squeeze().map(self.group_means_).fillna(self.global_mean_).values

# Pearson scorer
def pearsonr_scorer(y_true, y_pred):
	return pearsonr(y_true, y_pred)[0]

models = {
	'Random Forest': RandomForestRegressor(random_state=42),
	# 'XGBoost': XGBRegressor(objective='reg:squarederror', random_state=42),
	'Hist Gradient Boost': HistGradientBoostingRegressor(),
	'Group Mean': GroupMeanRegressor(),
	'Ridge': Ridge(alpha=1.0)
}
from sklearn.metrics import make_scorer
from scipy.stats import pearsonr

def pearsonr_scorer(y_true, y_pred):
	return pearsonr(y_true, y_pred)[0]

scoring_metrics = {
	'r2': 'r2',
	'neg_mae': 'neg_mean_absolute_error',
	'neg_rmse': 'neg_root_mean_squared_error',
}





In [5]:
## Investigate kurtosis and skewedness
print('Run on:', dt.datetime.today().strftime('%a %d %b %Y, %I:%M%p'))

skewed_cols = {}
for name in df_names:
	skewed_cols[name] = {}
	df = pd.read_csv(os.path.join(brighten_dir, f'{name}_trainval.csv'))
	
	print(f'\n\nFor {name}:')
	numeric_cols = df.select_dtypes(include=('int64','float64')).columns.to_list()
	non_bin_cols = [col for col in numeric_cols if '_bin' not in col and "_indicator" not in col and "_missing" not in col and "nonzero" not in col]
	
	skew_list = df[non_bin_cols].skew(numeric_only=True).sort_values(ascending=False) # sort by highest, display    
	if len(skew_list[skew_list > 1])>0:
		skewed_cols[name]['skew'] = skew_list[skew_list > 1].index
		print(f'Of {len(skew_list)} measures, {len(skew_list[skew_list > 1])} measures have skew > 1:')
		display(skew_list[skew_list > 1])

	# Calculate kurtosis for numeric columns
	kurtosis_vals = df[non_bin_cols].kurtosis(numeric_only=True)
	kurtosis_sorted = kurtosis_vals.sort_values(ascending=False) # Sort by highest kurtosis
	skewed_cols[name]['kurtosis'] = kurtosis_sorted[kurtosis_sorted > 2].index
	
	if len(kurtosis_sorted[kurtosis_sorted > 2])>0:
		print(f'Of {len(kurtosis_sorted)} measures, {len(kurtosis_sorted[kurtosis_sorted > 2])} measures have Kurtosis > 2:')
		display(kurtosis_sorted[kurtosis_sorted > 2]) #display

	if len(kurtosis_sorted[kurtosis_sorted.isna()])>0:
		print(f'{len(kurtosis_sorted[kurtosis_sorted.isna()])} columns have NaN in Kurtosis:')
		display(kurtosis_sorted[kurtosis_sorted.isna()]) #display

	# More investigation into kurtosis NaN values
	for col in kurtosis_sorted[kurtosis_sorted.isna()].index:
		print(f'Kurtosis is NaN for {col}:')
		if col in df.columns:
			print('Unique values:', df[col].nunique())      # Unique values
			print("Missing values:", df[col].isna().sum())   # Missing values
			print("Variance:", df[col].var())     # Summary stats



Run on: Wed 10 Jun 2026, 07:35PM


For v1_day:
Of 68 measures, 19 measures have skew > 1:


mobility_radius            20.239039
missed_interactions         6.321548
call_duration               5.960052
screen_4                    5.187388
sms_length                  4.844675
unreturned_calls            4.496510
sms_count                   4.352770
aggregate_communication     4.236915
screen_1                    3.970265
call_count                  3.225869
mobility                    2.698925
phq9_9                      2.218087
screen_2                    1.958110
phq9_9_base                 1.823661
phq9_8                      1.823628
screen_3                    1.822813
interaction_diversity       1.628570
season_num                  1.311941
gender                      1.141192
dtype: float64

Of 68 measures, 17 measures have Kurtosis > 2:


mobility_radius            469.780325
missed_interactions        111.767721
call_duration               54.747284
sms_length                  36.998472
unreturned_calls            33.726029
sms_count                   29.510155
aggregate_communication     28.719214
screen_4                    24.912594
call_count                  15.743431
mobility                    14.244014
screen_1                    13.764990
phq9_9                       4.634159
interaction_diversity        4.378336
num_id                       3.611457
phq9_9_base                  3.241968
phq9_8                       2.971215
season_num                   2.183329
dtype: float64

7 columns have NaN in Kurtosis:


...

Of 95 measures, 26 measures have Kurtosis > 2:


hours_active_hr                          314.746758
distance_high_speed_transportation       199.944066
hours_high_speed_transportation          196.154767
hours_high_speed_transportation_hr       147.188798
distance_high_speed_transportation_hr    137.758237
hours_powered_vehicle                    101.369840
hours_walking_hr                          64.677557
hours_active                              36.343310
distance_active                           29.954166
distance_powered_vehicle_hr               24.378635
distance_powered_vehicle                  21.900426
hours_walking                             18.048650
precip_sum                                17.552730
distance_walking_hr                       16.373391
hours_powered_vehicle_hr                  14.442255
screen_4                                   9.772097
distance_walking                           8.857497
location_variance                          6.720464
distance_active_hr                         5.242340
hours_of_sle

3 columns have NaN in Kurtosis:


cohort            NaN
user_phone_type   NaN
dt_phone_v1       NaN
dtype: float64

Kurtosis is NaN for cohort:
Unique values: 0
Missing values: 1648
Variance: nan
Kurtosis is NaN for user_phone_type:
Unique values: 0
Missing values: 1648
Variance: nan
Kurtosis is NaN for dt_phone_v1:
Unique values: 0
Missing values: 1648
Variance: nan


### Allocate column transformation strategy

In [3]:

print('Run on:', dt.datetime.today().strftime('%a %d %b %Y, %I:%M%p'))

target_columns = phq2_cols + phq9_cols
to_categorical = ['race','gender','marital_status', 'season', 'cohort']
ordinal_columns = list(set(gad_cols + phq9_base + alc_cols + sleep_cols + gic_cols + sds_cols +  ['education', 'income_satisfaction','incomelastyear'])) 

box_cox_columns = ['mobility','mobility_radius']
yeo_johnson_columns = list(set([col for col in skewed_cols[name]['skew'].to_list()+skewed_cols[name]['kurtosis'].to_list() if col not in box_cox_columns+target_columns+to_categorical+id_columns+ordinal_columns+baseline_cols and 'dt' not in col]))
non_skewed_columns = list(set([col for col in all_daily_cols if col not in box_cox_columns+target_columns+to_categorical+yeo_johnson_columns+ordinal_columns+id_columns and 'dt' not in col]))

def is_yj_safe(series):
	"""Check if a column is safe to apply Yeo-Johnson to."""
	s = series.dropna()
	if len(s) < 3:
		return False
	if s.nunique() < 3:  # nearly constant
		return False
	if s.std() == 0:  # zero variance
		return False
	return True


Run on: Wed 10 Jun 2026, 08:04PM


NameError: name 'skewed_cols' is not defined

# Do preprocessing column transformer on long data before making it wide

In [ ]:

for split in ['trainval','test']:
	for name in df_names:
		print(f"\n=== Processing: {name} ===")
		df = pd.read_csv(os.path.join(brighten_dir, f'{name}_{split}.csv'), low_memory=False)
		df = df.dropna(axis=1, how='all') # drop any columns which are fully NaN
		df['date'] = pd.to_datetime(df['date'])
		print('id columns originally:', [col for col in df.columns if col in id_columns])

		# Columns selected to each transformer
		cat_cols = [item for item in df.columns 
					if any(term in item for term in to_categorical) and 'dt' not in item]
		yj_cols = [
			item for item in df.columns 
			if any(term in item for term in yeo_johnson_columns) and 'dt' not in item
			and is_yj_safe(df[item])  # add this check
		]
		bc_cols = [item for item in df.columns 
				   if any(term in item for term in box_cox_columns) and 'dt' not in item]
		non_skewed_cols = [item for item in df.columns 
						   if any(term in item for term in non_skewed_columns) and 'dt' not in item]
		ordinal_cols = [col for col in df.columns 
						if any(term in col for term in ordinal_columns) and 'dt' not in col]
		target_cols = [col for col in df.columns 
					   if any(term in col for term in target_columns) and 'base' not in col and 'dt' not in col]


		for tname, cols in [('yj', yj_cols), ('bc', bc_cols), ('cat', cat_cols), ('non-skew', non_skewed_cols),('ordinal', ordinal_cols), ('target', target_cols)]:
			missing = [c for c in cols if c not in df.columns]
			if missing:
				print(f"Columns listed in {tname} but missing from df: {missing}")

			for col in cols:
				if col in df.columns:
					print(f'{tname} | {col} | dtype: {df[col].dtype} | sample: {df[col].dropna().iloc[0]}')



Run on: Fri 29 May 2026, 12:49PM

=== Processing: v1_day ===
id columns originally: ['day', 'date', 'week', 'v', 'cohort', 'num_id', 'id_day', 'id_week', 'day_of_week', 'month', 'season']
bc | mobility | dtype: float64 | sample: 6.653
bc | mobility_radius | dtype: float64 | sample: 6.125
cat | cohort | dtype: str | sample: PST
cat | gender | dtype: float64 | sample: 1.0
cat | marital_status | dtype: float64 | sample: 0.0
cat | race | dtype: float64 | sample: 2.0
cat | season | dtype: str | sample: summer
cat | season_num | dtype: float64 | sample: 4.0
non-skew | aggregate_communication | dtype: float64 | sample: 2.0
non-skew | call_count | dtype: float64 | sample: 2.0
non-skew | call_duration | dtype: float64 | sample: 33.0
non-skew | interaction_diversity | dtype: float64 | sample: 1.0
non-skew | missed_interactions | dtype: float64 | sample: 1.0
non-skew | sms_count | dtype: float64 | sample: 0.0
non-skew | sms_length | dtype: float64 | sample: 0.0
non-skew | unreturned_calls | dtype

In [4]:
for name in df_names:
	fitted_processor = None  # store fitted preprocessor here
	for split in ['trainval','test']:
		
		print(f"\n=== Processing: {name} ===")
		df = pd.read_csv(os.path.join(brighten_dir, f'{name}_{split}.csv'), low_memory=False)
		df['date'] = pd.to_datetime(df['date'])
		print('id columns originally:', [col for col in df.columns if col in id_columns])

		# Columns selected to each transformer
		cat_cols = [item for item in df.columns 
					if any(term in item for term in to_categorical) and 'dt' not in item]
		yj_cols = [
			item for item in df.columns 
			if any(term in item for term in yeo_johnson_columns) and 'dt' not in item
			and is_yj_safe(df[item])  # add this check
		]
		bc_cols = [item for item in df.columns 
				   if any(term in item for term in box_cox_columns) and 'dt' not in item]
		non_skewed_cols = [item for item in df.columns 
						   if any(term in item for term in non_skewed_columns) and 'dt' not in item]
		ordinal_cols = [col for col in df.columns 
						if any(term in col for term in ordinal_columns) and 'dt' not in col]
		target_cols = [col for col in df.columns 
					   if any(term in col for term in target_columns) and 'base' not in col and 'dt' not in col]
				
		transformers = []

		if len(yj_cols) > 0:
			transformers.append(('yj', yj_pipeline, yj_cols))
		if len(bc_cols) > 0:
			transformers.append(('bc', bc_pipeline, bc_cols))
		if len(cat_cols) > 0:
			transformers.append(('cat', categorical_pipeline, cat_cols))
		if len(non_skewed_cols) > 0:
			transformers.append(('non-skew', non_skewed_pipeline, non_skewed_cols))

		print(f'There are {len(transformers)} of 5 transformer pipelines')

		if split == 'trainval':
			# Fit only on training data
			fitted_preprocessor = ColumnTransformer(
				transformers=transformers,
				remainder='passthrough'
			).set_output(transform='pandas')
			fitted_preprocessor.fit(df)

		transformed_df = fitted_preprocessor.transform(df)

		transformed_df.columns = transformed_df.columns.str.replace('non-skew__', '', regex=False)
		transformed_df.columns = transformed_df.columns.str.replace('yj__', '', regex=False)
		transformed_df.columns = transformed_df.columns.str.replace('bc__', '', regex=False)
		transformed_df.columns = transformed_df.columns.str.replace('cat__', '', regex=False)
		transformed_df.columns = transformed_df.columns.str.replace('remainder__', '', regex=False)
		transformed_df = transformed_df.loc[:, ~transformed_df.columns.str.contains('_nan')]
		transformed_df = transformed_df.loc[:, ~transformed_df.columns.str.contains('Unnamed')]

		# Reorder id columns
		id_columns_intact = [col for col in id_columns if col in transformed_df.columns]
		transformed_df = transformed_df[id_columns_intact + [col for col in transformed_df.columns if col not in id_columns_intact]]

		# There's no one from this race in test_df and it's giving me problems with my predictive models later, and it's a column addded by cat encoder, so i just added it and said no one is this race by making it all = 0
		if 'test' in split:
			transformed_df['race_5.0'] = 0
			transformed_df['race_1.0'] = 0
			transformed_df['race_7.0'] = 0		
		

		transformed_df.to_csv(os.path.join(brighten_dir, f'{name}_{split}_nonskew.csv'))
		display(transformed_df)
		print(f'Saved {name}_{split}_nonskew.csv to brighten_dir')


print('Last run:', dt.datetime.today().strftime('%a %d %b %Y, %I:%M%p'))



=== Processing: v1_day ===
id columns originally: ['day', 'date', 'week', 'v', 'cohort', 'num_id', 'id_day', 'id_week', 'day_of_week', 'month', 'season']
There are 3 of 5 transformer pipelines


,num_id,date,month,week,day,id_week,id_day,v,day_of_week,mobility,...,screen_2,screen_3,screen_4,phq9_category,bin_clin,education,working,income_satisfaction,income_lastyear,age_category
0,1.0,2014-08-01,8.0,0,0,BLUE-00049_0,BLUE-00049_0,V1,5.0,NaN,...,0.0,0.0,0.0,med,0.0,2.0,0.0,3.0,6.0,3.0
1,1.0,2014-08-02,8.0,0,1,BLUE-00049_0,BLUE-00049_1,V1,6.0,2.339711,...,0.0,0.0,0.0,med,0.0,2.0,0.0,3.0,6.0,3.0
2,1.0,2014-08-03,8.0,0,2,BLUE-00049_0,BLUE-00049_2,V1,7.0,1.945768,...,0.0,0.0,0.0,med,0.0,2.0,0.0,3.0,6.0,3.0
3,1.0,2014-08-04,8.0,0,3,BLUE-00049_0,BLUE-00049_3,V1,1.0,2.216227,...,0.0,0.0,0.0,med,0.0,2.0,0.0,3.0,6.0,3.0
4,1.0,2014-08-05,8.0,0,4,BLUE-00049_0,BLUE-00049_4,V1,2.0,1.773836,...,0.0,0.0,0.0,med,0.0,2.0,0.0,3.0,6.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13939,1917.0,2015-01-16,1.0,10,75,YELLOW-00263_10,YELLOW-00263_75,V1,5.0,NaN,...,0.0,1.0,0.0,med,1.0,3.0,0.0,1.0,5.0,0.0
13940,1917.0,2015-01-17,1.0,10,76,YELLOW-00263_10,YELLOW-00263_76,V1,6.0,NaN,...,0.0,1.0,0.0,med,1.0,3.0,0.0,1.0,5.0,0.0
13941,1917.0,2015-01-17,1.0,11,77,YELLOW-00263_11,YELLOW-00263_77,V1,6.0,NaN,...,0.0,1.0,0.0,med,1.0,3.0,0.0,1.0,5.0,0.0
13942,1917.0,2015-01-18,1.0,11,78,YELLOW-00263_11,YELLOW-00263_78,V1,7.0,NaN,...,0.0,1.0,0.0,med,1.0,3.0,0.0,1.0,5.0,0.0


Saved v1_day_trainval_nonskew.csv to brighten_dir

=== Processing: v1_day ===
id columns originally: ['day', 'date', 'week', 'v', 'cohort', 'num_id', 'id_day', 'id_week', 'day_of_week', 'month', 'season']
There are 3 of 5 transformer pipelines


,num_id,date,month,week,day,id_week,id_day,v,day_of_week,mobility,...,screen_3,screen_4,phq9_category,bin_clin,education,working,income_satisfaction,income_lastyear,age_category,race_7.0
0,77.0,2014-10-14,10.0,0,0,BLUE-00129_0,BLUE-00129_0,V1,2.0,NaN,...,0.0,0.0,high,1.0,3.0,0.0,3.0,6.0,0.0,0
1,77.0,2014-10-14,10.0,0,1,BLUE-00129_0,BLUE-00129_1,V1,2.0,NaN,...,0.0,0.0,high,1.0,3.0,0.0,3.0,6.0,0.0,0
2,77.0,2014-10-15,10.0,0,2,BLUE-00129_0,BLUE-00129_2,V1,3.0,-1.119813,...,0.0,0.0,high,1.0,3.0,0.0,3.0,6.0,0.0,0
3,77.0,2014-10-15,10.0,0,3,BLUE-00129_0,BLUE-00129_3,V1,3.0,-1.119813,...,0.0,0.0,high,1.0,3.0,0.0,3.0,6.0,0.0,0
4,77.0,2014-10-16,10.0,0,4,BLUE-00129_0,BLUE-00129_4,V1,4.0,0.038154,...,0.0,0.0,high,1.0,3.0,0.0,3.0,6.0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2533,1914.0,2015-02-28,2.0,11,80,YELLOW-00260_11,YELLOW-00260_80,V1,6.0,NaN,...,0.0,0.0,med-low,0.0,4.0,1.0,1.0,5.0,2.0,0
2534,1914.0,2015-03-01,3.0,11,81,YELLOW-00260_11,YELLOW-00260_81,V1,7.0,NaN,...,0.0,0.0,med-low,0.0,4.0,1.0,1.0,5.0,2.0,0
2535,1914.0,2015-03-02,3.0,11,82,YELLOW-00260_11,YELLOW-00260_82,V1,1.0,NaN,...,0.0,0.0,med-low,0.0,4.0,1.0,1.0,5.0,2.0,0
2536,1914.0,2015-03-03,3.0,11,83,YELLOW-00260_11,YELLOW-00260_83,V1,2.0,NaN,...,0.0,0.0,med-low,0.0,4.0,1.0,1.0,5.0,2.0,0


Saved v1_day_test_nonskew.csv to brighten_dir

=== Processing: v2_day ===
id columns originally: ['day', 'date', 'week', 'v', 'cohort', 'num_id', 'id_day', 'id_week', 'day_of_week', 'month', 'season']
There are 3 of 5 transformer pipelines


...

Saved v1_week_test_nonskew.csv to brighten_dir

=== Processing: v2_week ===
id columns originally: ['id_week', 'day', 'date', 'week', 'v', 'cohort', 'num_id', 'day_of_week', 'month', 'season']
There are 3 of 5 transformer pipelines


,num_id,date,month,week,day,id_week,v,day_of_week,precip_sum,came_to_work,...,screen_2,screen_3,screen_4,phq9_category,bin_clin,education,working,income_satisfaction,income_lastyear,age_category
0,153.0,2016-08-13,8.000000,0,0,EN00033_0,V2,4.000000,1.519397,1.831350,...,NaN,NaN,NaN,med,1.0,4.0,1.0,2.0,0.0,3.0
1,153.0,2016-08-20,8.000000,1,7,EN00033_1,V2,4.000000,0.395146,1.598642,...,NaN,NaN,NaN,med,1.0,4.0,1.0,2.0,0.0,3.0
2,153.0,2016-08-27,8.285714,2,14,EN00033_2,V2,4.000000,0.214517,1.806629,...,NaN,NaN,NaN,med,1.0,4.0,1.0,2.0,0.0,3.0
3,153.0,2016-09-03,9.000000,3,21,EN00033_3,V2,4.000000,-0.659173,-0.591577,...,NaN,NaN,NaN,med,1.0,4.0,1.0,2.0,0.0,3.0
4,153.0,2016-09-10,9.000000,4,28,EN00033_4,V2,6.000000,-0.970441,-0.591577,...,NaN,NaN,NaN,med,1.0,4.0,1.0,2.0,0.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1643,1111.0,2017-03-05,3.000000,1,7,EN05370_1,V2,4.000000,-0.915956,-0.591577,...,NaN,NaN,NaN,high,1.0,3.0,0.0,0.0,2.0,1.0
1644,1111.0,2017-03-12,3.000000,2,14,EN05370_2,V2,4.000000,-0.970441,-0.591577,...,NaN,NaN,NaN,high,1.0,3.0,0.0,0.0,2.0,1.0
1645,1111.0,2017-03-19,3.000000,3,21,EN05370_3,V2,4.000000,-0.339663,-0.591577,...,NaN,NaN,NaN,high,1.0,3.0,0.0,0.0,2.0,1.0
1646,1111.0,2017-03-26,3.142857,4,28,EN05370_4,V2,4.000000,0.897720,-0.591577,...,NaN,NaN,NaN,high,1.0,3.0,0.0,0.0,2.0,1.0


Saved v2_week_trainval_nonskew.csv to brighten_dir

=== Processing: v2_week ===
id columns originally: ['id_week', 'day', 'date', 'week', 'v', 'cohort', 'num_id', 'day_of_week', 'month', 'season']
There are 3 of 5 transformer pipelines


,num_id,date,month,week,day,id_week,v,day_of_week,precip_sum,came_to_work,...,screen_3,screen_4,phq9_category,bin_clin,education,working,income_satisfaction,income_lastyear,age_category,race_5.0
0,190.0,2016-08-30,8.000000,0,0,EN00071_0,V2,2.000000,NaN,NaN,...,NaN,NaN,med-high,1.0,0.0,1.0,2.0,5.0,2.0,0
1,190.0,2016-09-06,9.000000,1,7,EN00071_1,V2,1.500000,1.030882,-0.591577,...,NaN,NaN,med-high,1.0,0.0,1.0,2.0,5.0,2.0,0
2,190.0,2016-11-08,11.000000,10,70,EN00071_10,V2,4.000000,-0.970441,-0.591577,...,NaN,NaN,med-high,1.0,0.0,1.0,2.0,5.0,2.0,0
3,190.0,2016-11-15,11.000000,11,77,EN00071_11,V2,4.000000,-0.563357,-0.591577,...,NaN,NaN,med-high,1.0,0.0,1.0,2.0,5.0,2.0,0
4,190.0,2016-11-22,11.000000,12,84,EN00071_12,V2,4.000000,0.925840,-0.591577,...,NaN,NaN,med-high,1.0,0.0,1.0,2.0,5.0,2.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
277,1040.0,2017-01-26,1.000000,5,35,EN05296_5,V2,5.714286,-0.970441,-0.591577,...,NaN,NaN,med-low,0.0,4.0,1.0,2.0,2.0,3.0,0
278,1040.0,2017-01-30,1.428571,6,42,EN05296_6,V2,2.285714,-0.970441,-0.591577,...,NaN,NaN,med-low,0.0,4.0,1.0,2.0,2.0,3.0,0
279,1040.0,2017-02-02,2.000000,7,49,EN05296_7,V2,5.714286,-0.215579,-0.591577,...,NaN,NaN,med-low,0.0,4.0,1.0,2.0,2.0,3.0,0
280,1040.0,2017-02-06,2.000000,8,56,EN05296_8,V2,2.285714,1.635127,-0.591577,...,NaN,NaN,med-low,0.0,4.0,1.0,2.0,2.0,3.0,0


Saved v2_week_test_nonskew.csv to brighten_dir
Last run: Fri 29 May 2026, 12:49PM


In [5]:
## Investigate kurtosis and skewedness
skewed_cols = {}
for name in df_names:
	skewed_cols[name] = {}
	df = pd.read_csv(os.path.join(brighten_dir, f'{name}_trainval_nonskew.csv'))
	print(f'\n\nFor {name}:')
	numeric_cols = [col for col in df.columns.to_list() if col in all_daily_cols+baseline_cols+weekly_cols+phq9_cols+phq2_cols]
	non_bin_cols = [col for col in numeric_cols if '_bin' not in col and "_indicator" not in col and "_missing" not in col and "nonzero" not in col]
	
	skew_list = df[non_bin_cols].skew(numeric_only=True).sort_values(ascending=False) # sort by highest, display    
	if len(skew_list[skew_list > 1])>0:
		skewed_cols[name]['skew'] = skew_list[skew_list > 1]
		print(f'Of {len(skew_list)} measures, {len(skew_list[skew_list > 1])} measures have skew > 1:')
		print(skew_list[skew_list > 1])

	# Calculate kurtosis for numeric columns
	kurtosis_vals = df[non_bin_cols].kurtosis(numeric_only=True)
	kurtosis_sorted = kurtosis_vals.sort_values(ascending=False) # Sort by highest kurtosis
	skewed_cols[name]['kurtosis'] = kurtosis_sorted[kurtosis_sorted > 2]
	if len(kurtosis_sorted[kurtosis_sorted > 2])>0:
		print(f'Of {len(kurtosis_sorted)} measures, {len(kurtosis_sorted[kurtosis_sorted > 2])} measures have Kurtosis > 2:')
		print(kurtosis_sorted[kurtosis_sorted > 2]) #display

	if len(kurtosis_sorted[kurtosis_sorted.isna()])>0:
		print(f'{len(kurtosis_sorted[kurtosis_sorted.isna()])} columns have NaN in Kurtosis:')
		print(kurtosis_sorted[kurtosis_sorted.isna()]) #display

	# More investigation into kurtosis NaN values
	for col in kurtosis_sorted[kurtosis_sorted.isna()].index:
		print(f'Kurtosis is NaN for {col}:')
		if col in df.columns:
			print('Unique values:', df[col].nunique())      # Unique values
			print("Missing values:", df[col].isna().sum())   # Missing values
			print("Variance:", df[col].var())     # Summary stats

print('Last run:', dt.datetime.today().strftime('%a %d %b %Y, %I:%M%p'))




For v1_day:
Of 54 measures, 17 measures have skew > 1:
missed_interactions        6.321548
call_duration              5.960052
screen_4                   5.187388
sms_length                 4.844675
unreturned_calls           4.496510
sms_count                  4.352770
aggregate_communication    4.236915
screen_1                   3.970265
call_count                 3.225869
cohort_PST                 3.081463
cohort_Akili               2.307262
phq9_9                     2.218087
screen_2                   1.958110
phq9_9_base                1.823661
phq9_8                     1.823628
screen_3                   1.822813
interaction_diversity      1.628570
dtype: float64
Of 54 measures, 15 measures have Kurtosis > 2:
missed_interactions        111.767721
call_duration               54.747284
sms_length                  36.998472
unreturned_calls            33.726029
sms_count                   29.510155
aggregate_communication     28.719214
screen_4                    24.912594
cal

# Standard Scale all numeric non-categorical columns

In [6]:
from sklearn.compose import ColumnTransformer
scaler = StandardScaler()
cols = 	yeo_johnson_columns+box_cox_columns+non_skewed_columns+ordinal_columns+target_columns


for name in df_names:
	fitted_processor = None  # store fitted preprocessor here
	for split in ['trainval','test']:
		skewed_cols[name] = {}
		df = pd.read_csv(os.path.join(brighten_dir, f'{name}_{split}_nonskew.csv'))
		cols_present = [col for col in cols if col in df.columns]
		df_cols_present = df[cols_present]
		df_cols_numeric = df[cols_present].select_dtypes(include=('int64','float64'))
		cols_not_numeric = [col for col in df_cols_present if col not in df_cols_numeric]
		if len(cols_not_numeric) > 0:
			print(f'Warning: these cols were not numeric: {cols_not_numeric}, skipping')
		
		if split=='trainval':
			fitted_processor = ColumnTransformer([('standard', scaler, df_cols_numeric.columns.to_list())], remainder = 'passthrough').set_output(transform='pandas')
			fitted_processor.fit(df)

		scaled_df = fitted_processor.transform(df)
		print(f'Scaled cols: {[col.replace('standard__','') for col in scaled_df.columns if 'standard' in col]}')
		scaled_df.columns = scaled_df.columns.str.replace('standard__', '')
		scaled_df.columns = scaled_df.columns.str.replace('remainder__', '')
		display(scaled_df)
		scaled_df.to_csv(os.path.join(brighten_dir, f'{name}_{split}_transformed.csv'), index=False)

print('Run on:', dt.datetime.today().strftime('%a %d %b %Y, %I:%M%p'))


Scaled cols: ['hours_accounted_for', 'hours_walking', 'mobility', 'mobility_radius', 'missed_interactions', 'call_duration', 'unreturned_calls', 'interaction_diversity', 'call_count', 'sms_length', 'aggregate_communication', 'sms_count', 'sleep_3', 'sleep_2', 'sds_2', 'phq9_8_base', 'education', 'phq9_2_base', 'stress', 'sleep_1', 'phq9_5_base', 'phq9_4_base', 'phq9_6_base', 'income_satisfaction', 'phq9_3_base', 'phq9_7_base', 'phq9_1_base', 'phq9_9_base', 'support', 'sds_1', 'sds_3', 'mood_1', 'phq2_1', 'phq2_2', 'phq2_sum', 'phq9_1', 'phq9_2', 'phq9_3', 'phq9_4', 'phq9_5', 'phq9_6', 'phq9_7', 'phq9_8', 'phq9_9', 'phq9_sum']


/Users/klj9278/miniconda3/envs/brighten/lib/python3.14/site-packages/sklearn/utils/extmath.py:1207: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/Users/klj9278/miniconda3/envs/brighten/lib/python3.14/site-packages/sklearn/utils/extmath.py:1212: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/Users/klj9278/miniconda3/envs/brighten/lib/python3.14/site-packages/sklearn/utils/extmath.py:1236: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count


,hours_accounted_for,hours_walking,mobility,mobility_radius,missed_interactions,call_duration,unreturned_calls,interaction_diversity,call_count,sms_length,...,dt_mobility_v2,screen_1,screen_2,screen_3,screen_4,phq9_category,bin_clin,working,income_lastyear,age_category
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.0,0.0,0.0,0.0,med,0.0,0.0,6.0,3.0
1,NaN,NaN,2.339711,0.435881,-0.110840,-0.458375,0.119001,-1.033784,-0.448868,-0.587252,...,NaN,0.0,0.0,0.0,0.0,med,0.0,0.0,6.0,3.0
2,NaN,NaN,1.945768,0.399027,-0.548976,-0.461154,-0.527078,-0.815694,-0.448868,-0.587252,...,NaN,0.0,0.0,0.0,0.0,med,0.0,0.0,6.0,3.0
3,NaN,NaN,2.216227,1.337419,-0.110840,-0.058875,-0.527078,0.056664,-0.318935,-0.460152,...,NaN,0.0,0.0,0.0,0.0,med,0.0,0.0,6.0,3.0
4,NaN,NaN,1.773836,1.342801,1.203566,0.271147,2.057239,-0.379515,0.460663,-0.523383,...,NaN,0.0,0.0,0.0,0.0,med,0.0,0.0,6.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13939,NaN,NaN,NaN,NaN,-0.548976,0.584495,-0.527078,0.274753,0.720530,0.027489,...,NaN,0.0,0.0,1.0,0.0,med,1.0,0.0,5.0,0.0
13940,NaN,NaN,NaN,NaN,0.765430,4.791061,-0.527078,2.019470,3.968858,-0.324430,...,NaN,0.0,0.0,1.0,0.0,med,1.0,0.0,5.0,0.0
13941,NaN,NaN,NaN,NaN,0.765430,4.791061,-0.527078,2.019470,3.968858,-0.324430,...,NaN,0.0,0.0,1.0,0.0,med,1.0,0.0,5.0,0.0
13942,NaN,NaN,NaN,NaN,-0.548976,0.110305,-0.527078,-0.161426,-0.189002,-0.552443,...,NaN,0.0,0.0,1.0,0.0,med,1.0,0.0,5.0,0.0


Scaled cols: ['hours_accounted_for', 'hours_walking', 'mobility', 'mobility_radius', 'missed_interactions', 'call_duration', 'unreturned_calls', 'interaction_diversity', 'call_count', 'sms_length', 'aggregate_communication', 'sms_count', 'sleep_3', 'sleep_2', 'sds_2', 'phq9_8_base', 'education', 'phq9_2_base', 'stress', 'sleep_1', 'phq9_5_base', 'phq9_4_base', 'phq9_6_base', 'income_satisfaction', 'phq9_3_base', 'phq9_7_base', 'phq9_1_base', 'phq9_9_base', 'support', 'sds_1', 'sds_3', 'mood_1', 'phq2_1', 'phq2_2', 'phq2_sum', 'phq9_1', 'phq9_2', 'phq9_3', 'phq9_4', 'phq9_5', 'phq9_6', 'phq9_7', 'phq9_8', 'phq9_9', 'phq9_sum']


,hours_accounted_for,hours_walking,mobility,mobility_radius,missed_interactions,call_duration,unreturned_calls,interaction_diversity,call_count,sms_length,...,dt_mobility_v2,screen_1,screen_2,screen_3,screen_4,phq9_category,bin_clin,working,income_lastyear,age_category
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.0,0.0,0.0,0.0,high,1.0,0.0,6.0,0.0
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.0,0.0,0.0,0.0,high,1.0,0.0,6.0,0.0
2,NaN,NaN,-1.119813,0.779476,-0.548976,-0.297880,-0.527078,-0.597605,-0.318935,-0.255132,...,NaN,0.0,0.0,0.0,0.0,high,1.0,0.0,6.0,0.0
3,NaN,NaN,-1.119813,0.779476,-0.548976,-0.297880,-0.527078,-0.597605,-0.318935,-0.255132,...,NaN,0.0,0.0,0.0,0.0,high,1.0,0.0,6.0,0.0
4,NaN,NaN,0.038154,0.851314,-0.548976,-0.072771,-0.527078,0.056664,0.070864,-0.382232,...,NaN,0.0,0.0,0.0,0.0,high,1.0,0.0,6.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2533,NaN,NaN,NaN,NaN,-0.548976,-0.469839,NaN,-0.379515,-0.708735,-0.531686,...,NaN,0.0,0.0,0.0,0.0,med-low,0.0,1.0,5.0,2.0
2534,NaN,NaN,NaN,NaN,-0.548976,-0.462544,-0.527078,-0.815694,-0.578802,-0.567772,...,NaN,0.0,0.0,0.0,0.0,med-low,0.0,1.0,5.0,2.0
2535,NaN,NaN,NaN,NaN,-0.548976,-0.419815,-0.527078,-1.033784,-0.578802,-0.587252,...,NaN,0.0,0.0,0.0,0.0,med-low,0.0,1.0,5.0,2.0
2536,NaN,NaN,NaN,NaN,-0.548976,-0.469839,NaN,-1.033784,-0.708735,-0.566814,...,NaN,0.0,0.0,0.0,0.0,med-low,0.0,1.0,5.0,2.0


...

Scaled cols: ['distance_powered_vehicle_hr', 'location_variance', 'hours_walking_hr', 'distance_high_speed_transportation_hr', 'precip_sum', 'location_variance_hr', 'came_to_work', 'distance_walking', 'hours_powered_vehicle', 'distance_high_speed_transportation', 'hours_accounted_for', 'hours_walking', 'hours_high_speed_transportation_hr', 'hours_of_sleep_hr', 'hours_active', 'hours_high_speed_transportation', 'distance_active', 'distance_walking_hr', 'distance_powered_vehicle', 'hours_active_hr', 'distance_active_hr', 'hours_powered_vehicle_hr', 'humidity_IQR', 'temp_median', 'dew_point_std', 'temp_mean', 'humidity_median', 'cloud_cover_IQR', 'dew_point_median', 'cloud_cover_mean', 'cloud_cover_median', 'hours_of_sleep', 'temp_std', 'temp_IQR', 'dew_point_mean', 'humidity_mean', 'dew_point_IQR', 'humidity_std', 'cloud_cover_std', 'sleep_3', 'sleep_2', 'sds_2', 'phq9_8_base', 'education', 'phq9_2_base', 'stress', 'sleep_1', 'phq9_5_base', 'phq9_4_base', 'phq9_6_base', 'income_satisfact

,distance_powered_vehicle_hr,location_variance,hours_walking_hr,distance_high_speed_transportation_hr,precip_sum,location_variance_hr,came_to_work,distance_walking,hours_powered_vehicle,distance_high_speed_transportation,...,dt_mobility_v2,screen_1,screen_2,screen_3,screen_4,phq9_category,bin_clin,working,income_lastyear,age_category
0,0.512344,-0.213285,-0.513301,2.169585,1.519397,0.899558,1.831350,0.279157,-0.485520,2.069189,...,2016-08-13,NaN,NaN,NaN,NaN,med,1.0,1.0,0.0,3.0
1,0.681278,0.700174,1.409264,-0.462517,0.395146,0.843865,1.598642,1.095057,0.433552,-0.478665,...,2016-08-20,NaN,NaN,NaN,NaN,med,1.0,1.0,0.0,3.0
2,-2.029004,-0.103298,0.600106,-0.462517,0.214517,-0.463238,1.806629,-0.200794,-1.541384,-0.478665,...,2016-08-28,NaN,NaN,NaN,NaN,med,1.0,1.0,0.0,3.0
3,1.889570,0.896299,1.650790,2.157567,-0.659173,0.920340,-0.591577,1.010552,1.356401,2.071635,...,2016-09-03,NaN,NaN,NaN,NaN,med,1.0,1.0,0.0,3.0
4,NaN,NaN,NaN,NaN,-0.970441,NaN,-0.591577,-2.227640,-1.667372,-0.478665,...,2016-09-10,NaN,NaN,NaN,NaN,med,1.0,1.0,0.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1643,1.152436,0.974020,-0.222140,-0.462517,-0.915956,0.817658,-0.591577,0.300528,1.153611,-0.478665,...,2017-03-05,NaN,NaN,NaN,NaN,high,1.0,0.0,2.0,1.0
1644,1.004668,0.703855,-0.858545,-0.462517,-0.970441,-0.000463,-0.591577,-0.605818,0.978234,-0.478665,...,2017-03-12,NaN,NaN,NaN,NaN,high,1.0,0.0,2.0,1.0
1645,1.010281,0.873558,1.557012,-0.462517,-0.339663,0.417958,-0.591577,0.681691,0.999631,-0.478665,...,2017-03-19,NaN,NaN,NaN,NaN,high,1.0,0.0,2.0,1.0
1646,1.767462,0.863629,0.614380,-0.462517,0.897720,0.438614,-0.591577,0.493703,1.223123,-0.478665,...,2017-03-26,NaN,NaN,NaN,NaN,high,1.0,0.0,2.0,1.0


Scaled cols: ['distance_powered_vehicle_hr', 'location_variance', 'hours_walking_hr', 'distance_high_speed_transportation_hr', 'precip_sum', 'location_variance_hr', 'came_to_work', 'distance_walking', 'hours_powered_vehicle', 'distance_high_speed_transportation', 'hours_accounted_for', 'hours_walking', 'hours_high_speed_transportation_hr', 'hours_of_sleep_hr', 'hours_active', 'hours_high_speed_transportation', 'distance_active', 'distance_walking_hr', 'distance_powered_vehicle', 'hours_active_hr', 'distance_active_hr', 'hours_powered_vehicle_hr', 'humidity_IQR', 'temp_median', 'dew_point_std', 'temp_mean', 'humidity_median', 'cloud_cover_IQR', 'dew_point_median', 'cloud_cover_mean', 'cloud_cover_median', 'hours_of_sleep', 'temp_std', 'temp_IQR', 'dew_point_mean', 'humidity_mean', 'dew_point_IQR', 'humidity_std', 'cloud_cover_std', 'sleep_3', 'sleep_2', 'sds_2', 'phq9_8_base', 'education', 'phq9_2_base', 'stress', 'sleep_1', 'phq9_5_base', 'phq9_4_base', 'phq9_6_base', 'income_satisfact

,distance_powered_vehicle_hr,location_variance,hours_walking_hr,distance_high_speed_transportation_hr,precip_sum,location_variance_hr,came_to_work,distance_walking,hours_powered_vehicle,distance_high_speed_transportation,...,dt_mobility_v2,screen_1,screen_2,screen_3,screen_4,phq9_category,bin_clin,working,income_lastyear,age_category
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,med-high,1.0,1.0,5.0,2.0
1,1.684259,1.525296,-0.810378,-0.462517,1.030882,1.729484,-0.591577,-0.340501,1.949972,-0.478665,...,2016-09-12,NaN,NaN,NaN,NaN,med-high,1.0,1.0,5.0,2.0
2,1.228479,0.537551,0.377922,-0.462517,-0.970441,0.064381,-0.591577,0.265657,1.109622,-0.478665,...,2016-11-08,NaN,NaN,NaN,NaN,med-high,1.0,1.0,5.0,2.0
3,1.017199,0.851187,1.266590,-0.462517,-0.563357,0.961608,-0.591577,1.378299,1.514803,-0.478665,...,2016-11-15,NaN,NaN,NaN,NaN,med-high,1.0,1.0,5.0,2.0
4,1.414342,0.652230,0.787822,-0.462517,0.925840,0.439802,-0.591577,0.427885,1.289907,-0.478665,...,2016-11-22,NaN,NaN,NaN,NaN,med-high,1.0,1.0,5.0,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
277,0.907768,1.085209,-1.017030,-0.462517,-0.970441,1.077489,-0.591577,-0.197127,1.506110,-0.478665,...,2017-01-26,NaN,NaN,NaN,NaN,med-low,0.0,1.0,2.0,3.0
278,0.670735,0.687194,0.974736,-0.462517,-0.970441,0.413446,-0.591577,0.881809,1.141098,-0.478665,...,2017-01-30,NaN,NaN,NaN,NaN,med-low,0.0,1.0,2.0,3.0
279,1.768085,1.704352,0.457687,2.178586,-0.215579,1.954598,-0.591577,0.825564,2.178435,2.104366,...,2017-02-02,NaN,NaN,NaN,NaN,med-low,0.0,1.0,2.0,3.0
280,1.324220,1.027571,0.703050,-0.462517,1.635127,0.819757,-0.591577,0.730533,1.822241,-0.478665,...,2017-02-06,NaN,NaN,NaN,NaN,med-low,0.0,1.0,2.0,3.0


Run on: Fri 29 May 2026, 12:49PM


# Make imputed version of the dataset
People are already filtered to have >70% of data
Imputing their mean so that we can use PCA and other non-NA methods

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='mean', missing_values=np.nan)

print('Run on:', dt.datetime.today().strftime('%a %d %b %Y, %I:%M%p'))

numeric_cols = 	yeo_johnson_columns+box_cox_columns+non_skewed_columns+ordinal_columns+target_columns

imputed_df = pd.DataFrame()
for name in df_names:
	count=0
	for split in ['trainval','test']:
		df = pd.read_csv(os.path.join(brighten_dir, f'{name}_{split}_transformed.csv'))
		numeric_cols_present = [col for col in numeric_cols if col in df.columns]
		df_cols_present = df[numeric_cols_present]
		df_cols_numeric = df[numeric_cols_present].select_dtypes(include=('int64','float64'))
		cols_not_numeric = [col for col in df_cols_present if col not in df_cols_numeric]
		if len(cols_not_numeric) > 0:
			print(f'Warning: these cols were not numeric: {cols_not_numeric}, skipping')

		# Skip imputing phq9 cols so we don't impute our target...
		cols_to_impute = [col for col in df_cols_numeric.columns if col not in phq9_cols]
		imputer_pipe = ColumnTransformer([('impute', imputer, cols_to_impute)], remainder = 'passthrough').set_output(transform='pandas')

		all_imputed = []  # collect per-subject results

		for count, (sub, sub_df) in enumerate(df.groupby('num_id')):
			sub_df.to_csv(os.path.join(paths.SUB_DFS, f'{sub}_trainval_transformed.csv'))
			imputed_sub_df = imputer_pipe.fit_transform(sub_df)

			# clean up column names once per loop
			imputed_sub_df.columns = imputed_sub_df.columns.str.replace('impute__', '', regex=False)
			imputed_sub_df.columns = imputed_sub_df.columns.str.replace('remainder__', '', regex=False)

			imputed_sub_df['num_id'] = sub
			imputed_sub_df.to_csv(os.path.join(paths.SUB_DFS, f'{sub}_trainval_imputed.csv'))
			all_imputed.append(imputed_sub_df)
			

			if count == 0:
				imputed_cols = [col for col in imputed_sub_df.columns if col in cols_to_impute and col not in baseline_cols]
				print(f'Imputed cols for {name}: {imputed_cols}')
				count+=1


		# concatenate all subjects for this file
		imputed_df = pd.concat(all_imputed, ignore_index=True)

		display(imputed_df[[col for col in id_columns if col in imputed_df.columns]+[col for col in imputed_df.columns if col not in id_columns]].head())

		# save output
		out_path = os.path.join(brighten_dir, f'{name}_{split}_imputed.csv')
		imputed_df.to_csv(out_path, index=False)
		print(f'Saved imputed data to {out_path}')



Run on: Wed 10 Jun 2026, 07:49PM
Imputed cols for v1_day: ['mobility', 'mobility_radius', 'call_duration', 'interaction_diversity', 'sms_length', 'sms_count', 'unreturned_calls', 'aggregate_communication', 'call_count', 'missed_interactions', 'support', 'sds_1', 'sds_2', 'sleep_2', 'stress', 'sleep_1', 'sleep_3', 'sds_3', 'phq2_1', 'phq2_2', 'phq2_sum']


,num_id,date,month,week,day,id_week,id_day,v,day_of_week,mobility,...,screen_1,screen_2,screen_3,screen_4,phq9_category,bin_clin,working,income_lastyear,age_category,mood_1
0,1.0,2014-08-01,8.0,0,0,BLUE-00049_0,BLUE-00049_0,V1,5.0,1.679190,...,0.0,0.0,0.0,0.0,med,0.0,0.0,6.0,3.0,NaN
1,1.0,2014-08-02,8.0,0,1,BLUE-00049_0,BLUE-00049_1,V1,6.0,2.339711,...,0.0,0.0,0.0,0.0,med,0.0,0.0,6.0,3.0,NaN
2,1.0,2014-08-03,8.0,0,2,BLUE-00049_0,BLUE-00049_2,V1,7.0,1.945768,...,0.0,0.0,0.0,0.0,med,0.0,0.0,6.0,3.0,NaN
3,1.0,2014-08-04,8.0,0,3,BLUE-00049_0,BLUE-00049_3,V1,1.0,2.216227,...,0.0,0.0,0.0,0.0,med,0.0,0.0,6.0,3.0,NaN
4,1.0,2014-08-05,8.0,0,4,BLUE-00049_0,BLUE-00049_4,V1,2.0,1.773836,...,0.0,0.0,0.0,0.0,med,0.0,0.0,6.0,3.0,NaN


Saved imputed data to /Users/klj9278/Library/CloudStorage/Box-Box/Kaley_research_NYU/EMA_projects/smartphone_sensor_modelling_may26/data/interim/v1_day_trainval_imputed.csv
Imputed cols for v1_day: ['mobility', 'mobility_radius', 'call_duration', 'interaction_diversity', 'sms_length', 'sms_count', 'unreturned_calls', 'aggregate_communication', 'call_count', 'missed_interactions', 'support', 'mood_1', 'sds_1', 'sds_2', 'sleep_2', 'stress', 'sleep_1', 'sleep_3', 'sds_3', 'phq2_1', 'phq2_2', 'phq2_sum']


,num_id,date,month,week,day,id_week,id_day,v,day_of_week,mobility,...,dt_mobility_v2,screen_1,screen_2,screen_3,screen_4,phq9_category,bin_clin,working,income_lastyear,age_category
0,77.0,2014-10-14,10.0,0,0,BLUE-00129_0,BLUE-00129_0,V1,2.0,0.042224,...,NaN,0.0,0.0,0.0,0.0,high,1.0,0.0,6.0,0.0
1,77.0,2014-10-14,10.0,0,1,BLUE-00129_0,BLUE-00129_1,V1,2.0,0.042224,...,NaN,0.0,0.0,0.0,0.0,high,1.0,0.0,6.0,0.0
2,77.0,2014-10-15,10.0,0,2,BLUE-00129_0,BLUE-00129_2,V1,3.0,-1.119813,...,NaN,0.0,0.0,0.0,0.0,high,1.0,0.0,6.0,0.0
3,77.0,2014-10-15,10.0,0,3,BLUE-00129_0,BLUE-00129_3,V1,3.0,-1.119813,...,NaN,0.0,0.0,0.0,0.0,high,1.0,0.0,6.0,0.0
4,77.0,2014-10-16,10.0,0,4,BLUE-00129_0,BLUE-00129_4,V1,4.0,0.038154,...,NaN,0.0,0.0,0.0,0.0,high,1.0,0.0,6.0,0.0


Saved imputed data to /Users/klj9278/Library/CloudStorage/Box-Box/Kaley_research_NYU/EMA_projects/smartphone_sensor_modelling_may26/data/interim/v1_day_test_imputed.csv
Imputed cols for v2_day: ['hours_powered_vehicle', 'distance_walking_hr', 'hours_active_hr', 'distance_active', 'distance_powered_vehicle', 'hours_accounted_for', 'hours_high_speed_transportation_hr', 'hours_walking_hr', 'hours_active', 'location_variance', 'hours_walking', 'distance_high_speed_transportation_hr', 'hours_powered_vehicle_hr', 'precip_sum', 'distance_high_speed_transportation', 'distance_active_hr', 'location_variance_hr', 'distance_walking', 'hours_high_speed_transportation', 'distance_powered_vehicle_hr', 'hours_of_sleep_hr', 'came_to_work', 'hours_stationary_nhw', 'cloud_cover_mean', 'cloud_cover_IQR', 'cloud_cover_std', 'dew_point_mean', 'dew_point_std', 'dew_point_median', 'cloud_cover_median', 'dew_point_IQR', 'temp_IQR', 'humidity_IQR', 'temp_mean', 'humidity_std', 'temp_std', 'hours_stationary', '

...

Saved imputed data to /Users/klj9278/Library/CloudStorage/Box-Box/Kaley_research_NYU/EMA_projects/smartphone_sensor_modelling_may26/data/interim/v1_week_test_imputed.csv
Imputed cols for v2_week: ['hours_powered_vehicle', 'distance_walking_hr', 'hours_active_hr', 'distance_active', 'distance_powered_vehicle', 'hours_accounted_for', 'hours_high_speed_transportation_hr', 'hours_walking_hr', 'hours_active', 'location_variance', 'hours_walking', 'distance_high_speed_transportation_hr', 'hours_powered_vehicle_hr', 'precip_sum', 'distance_high_speed_transportation', 'distance_active_hr', 'location_variance_hr', 'distance_walking', 'hours_high_speed_transportation', 'distance_powered_vehicle_hr', 'hours_of_sleep_hr', 'came_to_work', 'hours_stationary_nhw', 'cloud_cover_mean', 'cloud_cover_IQR', 'cloud_cover_std', 'dew_point_mean', 'dew_point_std', 'dew_point_median', 'cloud_cover_median', 'dew_point_IQR', 'temp_IQR', 'humidity_IQR', 'temp_mean', 'humidity_std', 'temp_std', 'hours_stationary',

,num_id,date,month,week,day,id_week,v,day_of_week,hours_powered_vehicle,distance_walking_hr,...,sds_1,sds_2,sleep_2,stress,sleep_1,sleep_3,sds_3,phq2_1,phq2_2,phq2_sum
0,153.0,2016-08-13,8.000000,0,0,EN00033_0,V2,4.0,-0.485520,-0.097734,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,153.0,2016-08-20,8.000000,1,7,EN00033_1,V2,4.0,0.433552,1.305295,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,153.0,2016-08-27,8.285714,2,14,EN00033_2,V2,4.0,-1.541384,0.692996,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,153.0,2016-09-03,9.000000,3,21,EN00033_3,V2,4.0,1.356401,1.500550,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,153.0,2016-09-10,9.000000,4,28,EN00033_4,V2,6.0,-1.667372,0.850277,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Saved imputed data to /Users/klj9278/Library/CloudStorage/Box-Box/Kaley_research_NYU/EMA_projects/smartphone_sensor_modelling_may26/data/interim/v2_week_trainval_imputed.csv
Imputed cols for v2_week: ['hours_powered_vehicle', 'distance_walking_hr', 'hours_active_hr', 'distance_active', 'distance_powered_vehicle', 'hours_accounted_for', 'hours_high_speed_transportation_hr', 'hours_walking_hr', 'hours_active', 'location_variance', 'hours_walking', 'distance_high_speed_transportation_hr', 'hours_powered_vehicle_hr', 'precip_sum', 'distance_high_speed_transportation', 'distance_active_hr', 'location_variance_hr', 'distance_walking', 'hours_high_speed_transportation', 'distance_powered_vehicle_hr', 'hours_of_sleep_hr', 'came_to_work', 'hours_stationary_nhw', 'cloud_cover_mean', 'cloud_cover_IQR', 'cloud_cover_std', 'dew_point_mean', 'dew_point_std', 'dew_point_median', 'cloud_cover_median', 'dew_point_IQR', 'temp_IQR', 'humidity_IQR', 'temp_mean', 'humidity_std', 'temp_std', 'hours_stationa

,num_id,date,month,week,day,id_week,v,day_of_week,hours_powered_vehicle,distance_walking_hr,...,age_category,support,mood_1,sds_1,sds_2,sleep_2,stress,sleep_1,sleep_3,sds_3
0,190.0,2016-08-30,8.0,0,0,EN00071_0,V2,2.0,1.092221,0.089985,...,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,190.0,2016-09-06,9.0,1,7,EN00071_1,V2,1.5,1.949972,-0.597613,...,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,190.0,2016-11-08,11.0,10,70,EN00071_10,V2,4.0,1.109622,0.362202,...,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,190.0,2016-11-15,11.0,11,77,EN00071_11,V2,4.0,1.514803,1.088851,...,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,190.0,2016-11-22,11.0,12,84,EN00071_12,V2,4.0,1.289907,0.852760,...,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Saved imputed data to /Users/klj9278/Library/CloudStorage/Box-Box/Kaley_research_NYU/EMA_projects/smartphone_sensor_modelling_may26/data/interim/v2_week_test_imputed.csv


# Explore raw vs. imputed data for subjects in V1

In [27]:
import pandas as pd
import numpy as np
import os
import plotly.express as px
from dash import Dash, html, dcc, callback, Output, Input
from urllib.request import urlopen
import json


app = Dash()

split='trainval'
name='v1_day'
df_raw = pd.read_csv(os.path.join(brighten_dir, f'{name}_{split}_transformed.csv'))
df_imp = pd.read_csv(os.path.join(brighten_dir, f'{name}_{split}_imputed.csv'))

subs = list(df_raw['num_id'].unique())
features = [col for col in df_raw.columns if col not in id_columns+baseline_cols+['hours_accounted_for','hours_walking','hours_stationary','hours_stationary_nhw']]
print(features)

app.layout = html.Div([
	dcc.Dropdown(options=subs, value=subs[0], id='sub-selected'),
	dcc.Dropdown(options=features, value=features[0], id='feature-selected'),
	dcc.Graph(figure={}, id='fig-raw'),
	dcc.Graph(figure={}, id='fig-imp')	
	])



@callback(
	Output(component_id='fig-raw', component_property='figure'),
	Output(component_id='fig-imp', component_property='figure'),
	Input(component_id='sub-selected', component_property='value'),
	Input(component_id='feature-selected', component_property='value'),

)

def update_graph(sub, feature):
	raw = df_raw.query("num_id==@sub")
	raw = raw.sort_values(by='day')
	fig_raw = px.line(raw[['num_id','day',feature]], x='day', y=feature, title=f'Raw {feature} Values for Sub {sub}', height=300, width=400, markers='o')
	imp = df_imp.query("num_id==@sub")
	imp = imp.sort_values(by='day')
	fig_imp = px.line(imp[['num_id','day',feature]], x='day', y=feature, title=f'Imputed {feature} Values for Sub {sub}', height=300, width=400, markers='o')
				
	return fig_raw, fig_imp
	
if __name__ == '__main__':
	app.run(debug=True, port=5022)


['mobility', 'mobility_radius', 'missed_interactions', 'call_duration', 'unreturned_calls', 'interaction_diversity', 'call_count', 'sms_length', 'aggregate_communication', 'sms_count', 'sleep_3', 'sleep_2', 'sds_2', 'stress', 'sleep_1', 'support', 'sds_1', 'sds_3', 'mood_1', 'phq2_1', 'phq2_2', 'phq2_sum', 'phq9_1', 'phq9_2', 'phq9_3', 'phq9_4', 'phq9_5', 'phq9_6', 'phq9_7', 'phq9_8', 'phq9_9', 'phq9_sum', 'Unnamed: 0', 'gender_0.0', 'gender_1.0', 'marital_status_0.0', 'marital_status_1.0', 'marital_status_2.0', 'race_0.0', 'race_1.0', 'race_2.0', 'race_3.0', 'race_4.0', 'race_5.0', 'race_6.0', 'season_fall', 'season_spring', 'season_summer', 'season_winter', 'season_num_1.0', 'season_num_2.0', 'season_num_3.0', 'season_num_4.0', 'participant_id', 'phq9_cat', 'phq9_bin', 'dt_phq9', 'phq2_bin', 'dt_phq2', 'dt_sds', 'dt_sleep', 'dt_gic', 'dt_phone_v1', 'dt_phone_v2', 'dt_weather_v2', 'dt_mobility_v2', 'phq9_category', 'age_category']


In [26]:
import pandas as pd
import numpy as np
import os
import plotly.express as px
from dash import Dash, html, dcc, callback, Output, Input
from urllib.request import urlopen
import json


app = Dash()

split='trainval'
name='v2_day'
df_raw = pd.read_csv(os.path.join(brighten_dir, f'{name}_{split}_transformed.csv'))
df_imp = pd.read_csv(os.path.join(brighten_dir, f'{name}_{split}_imputed.csv'))

subs = list(df_raw['num_id'].unique())
features = [col for col in df_raw.columns if col not in id_columns+baseline_cols+['hours_accounted_for','hours_walking']]
print(features)
app.layout = html.Div([
	dcc.Dropdown(options=subs, value=subs[0], id='sub-selected'),
	dcc.Dropdown(options=features, value=features[0], id='feature-selected'),
	dcc.Graph(figure={}, id='fig-raw'),
	dcc.Graph(figure={}, id='fig-imp')	
	])



@callback(
	Output(component_id='fig-raw', component_property='figure'),
	Output(component_id='fig-imp', component_property='figure'),
	Input(component_id='sub-selected', component_property='value'),
	Input(component_id='feature-selected', component_property='value'),

)

def update_graph(sub, feature):
	raw = df_raw.query("num_id==@sub")
	raw = raw.sort_values(by='day')
	fig_raw = px.line(raw[['num_id','day',feature]], x='day', y=feature, title=f'Raw {feature} Values for Sub {sub}', height=300, width=400, markers='o')
	imp = df_imp.query("num_id==@sub")
	imp = imp.sort_values(by='day')
	fig_imp = px.line(imp[['num_id','day',feature]], x='day', y=feature, title=f'Imputed {feature} Values for Sub {sub}', height=300, width=400, markers='o')
				
	return fig_raw, fig_imp
	
if __name__ == '__main__':
	app.run(debug=True, port=5023)


['distance_powered_vehicle_hr', 'location_variance', 'hours_walking_hr', 'distance_high_speed_transportation_hr', 'precip_sum', 'location_variance_hr', 'came_to_work', 'distance_walking', 'hours_powered_vehicle', 'distance_high_speed_transportation', 'hours_high_speed_transportation_hr', 'hours_of_sleep_hr', 'hours_active', 'hours_high_speed_transportation', 'distance_active', 'distance_walking_hr', 'distance_powered_vehicle', 'hours_active_hr', 'distance_active_hr', 'hours_powered_vehicle_hr', 'humidity_IQR', 'temp_median', 'dew_point_std', 'temp_mean', 'humidity_median', 'cloud_cover_IQR', 'dew_point_median', 'cloud_cover_mean', 'cloud_cover_median', 'hours_of_sleep', 'temp_std', 'temp_IQR', 'dew_point_mean', 'humidity_mean', 'dew_point_IQR', 'humidity_std', 'cloud_cover_std', 'sleep_3', 'sleep_2', 'sds_2', 'stress', 'sleep_1', 'support', 'sds_1', 'sds_3', 'mood_1', 'phq2_1', 'phq2_2', 'phq2_sum', 'phq9_1', 'phq9_2', 'phq9_3', 'phq9_4', 'phq9_5', 'phq9_6', 'phq9_7', 'phq9_8', 'phq9_9